# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zainab-Aijaz/WEEK1_ML_Assignment_FlyRank_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
from huggingface_hub import login

login()

In [6]:
from google.colab import userdata
import os

os.environ["HF-TOKEN"] = userdata.get("HF-TOKEN")

In [7]:
from datasets import load_dataset
import duckdb


fact_content_ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", split="train")
fact_content = fact_content_ds.data.table   # Arrow, not pandas — much lighter

dim_content_ds = load_dataset("FlyRank/internship-warehouse", "dim_content", split="train")
dim_content = dim_content_ds.data.table

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 21.6MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  624kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 86.2MB            

...

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

In [4]:
from google.colab import userdata
import os

token = userdata.get("HF-TOKEN")
print("Token loaded:", token is not None, "| length:", len(token) if token else 0)

Token loaded: True | length: 37


In [ ]:
from google.colab import userdata
import duckdb

token = userdata.get("HF-TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{token}'
)
""") # accept the gate in-browser first, then paste a READ token
rel = "hf://datasets/FlyRank/internship-warehouse"
con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

In [ ]:

# grain check on dim_content — is it really one row per content+client?
duckdb.sql("""
    SELECT client_hash_id, content_hash_id, COUNT(*) c
    FROM dim_content
    GROUP BY client_hash_id, content_hash_id
    HAVING c > 1
    LIMIT 5
""")
# if rows come back, a content_hash_id maps to >1 keyword/url — the real grain is
# content+client+keyword, not content+client. Check before assuming.

In [ ]:



# grain check on fact table
duckdb.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM fact_content
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""")
# expect 0 rows back

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## **Feature (knowable before the decision moment, from dim_content):**

 *content_type, search_volume, competition, competition_level, cpc, main_intent, backlinks, category_count, char_count, word_count, content_created_date, content_updated_date, last_optimized_date, optimization_eligible_date, is_published.*


##**Feature, rolled up from fact_content_daily_performance**

*— but only using days strictly BEFORE the decision date: prior gsc_clicks, gsc_impressions, gsc_avg_position, ga4_sessions, ga4_engaged_sessions trends.*

##**Label / proxy:**

*whether gsc_clicks decline over the 30 days after the decision date — computed only from days after, never before.*

##**Context**

 (for joining/grouping/filtering, never fed to the model): client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, report_date.

## **Excluded, with why:**

* provider_used, model_used — describes who/what authored the content internally, not a real signal about page performance.


* is_deleted — a lifecycle/product-decision flag, not a performance signal; deleted pages shouldn't be in the scoring pool at all.


* ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other — too granular per-AI-source breakdown for this lane; sessions_ai at the aggregate level is enough, and using the same-window breakdown risks overlapping with the label window.


* client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available — these are availability flags, used to filter rows, not fed to the model as predictive features.





## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# counts + date span for the month
duckdb.sql("""
    SELECT COUNT(*) AS total_rows, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""")

# availability: how many rows actually have usable GSC data
duckdb.sql("""
    SELECT COUNT(*) AS available_rows
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
""")

# missingness overall
duckdb.sql("""
    SELECT
      AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position,
      AVG(CASE WHEN ga4_sessions IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_ga4_sessions
    FROM fact_content
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
""")

# missingness by content_type (patterned gaps check)
duckdb.sql("""
    SELECT d.content_type,
           AVG(CASE WHEN f.gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) AS pct_missing_position
    FROM fact_content f
    JOIN dim_content d USING (client_hash_id, content_hash_id)
    WHERE f.report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY d.content_type
""")

# windows per client — histories rarely start together
duckdb.sql("""
    SELECT client_hash_id, MIN(report_date) AS first_date, MAX(report_date) AS last_date
    FROM fact_content
    GROUP BY client_hash_id
    LIMIT 10
""")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

# **Data limits**

This data can never tell me whether a refresh caused recovery — no page here was refreshed and observed afterward, so "declining → needs refresh" stays a proxy, not proven cause and effect.
client_has_gsc / client_has_ga4 show that not every client has both data sources — a client without GA4 will have ga4_sessions, ga4_engaged_sessions, etc. as structurally missing, not zero engagement. Any feature built from GA4 needs to check this flag first, or it'll misread "no tracking" as "no engagement."
content_created_date vs keyword_created_date may not align — a page could be older than the keyword record or vice versa, so content age needs to be computed from the right date, not assumed.
Clients don't share a common history start — comparing "declining in March" across clients isn't apples-to-apples unless each client's own data start date is checked first (see the windows-per-client query above)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.